# Deploying FLUX.2 [klein] 9B on SageMaker Async Inference

Async endpoint serving Black Forest Labs' [FLUX.2 [klein] 9B](https://huggingface.co/black-forest-labs/FLUX.2-klein-9B) (bf16 weights, with CPU offload to fit comfortably on an L40S). One endpoint handles both text-to-image generation and multi-reference image editing.

**Deployment shape:**
- Weights live as **uncompressed objects** under an S3 prefix. SageMaker mounts them at `/opt/ml/model/` on the container. No tarball, no re-archiving when you redeploy.
- The `code/` directory is uploaded **separately** as a tiny `sourcedir.tar.gz` on every deploy. Edit the handler, redeploy in seconds without touching the 33 GB of weights.
- Container env carries no `HF_TOKEN`. The Hub is only touched once during weight preparation.

**Sequence:**
1. One-time: run `prepare_weights.py` locally (or from this notebook) to download the bf16 weights.
2. One-time: `aws s3 sync` those weights to a stable S3 prefix.
3. Per deploy: this notebook builds the `PyTorchModel` with `model_data` pointing at the prefix, and ships `code/` as the source bundle.


## 1. Pin the SageMaker SDK


In [ ]:
!pip uninstall -y sagemaker
!pip install "sagemaker<3.0.0"

## 1b. Choose the GPU instance

Pick the instance family to deploy on. This drives the instance type, the
endpoint name suffix, and the inference memory strategy.

- **g6e** (`ml.g6e.2xlarge`, NVIDIA L40S 48 GB) — **supported today.** bf16
  weights are tight, so the handler uses `enable_model_cpu_offload()`
  (JIT CPU<->GPU swaps).
- **g7e** (`ml.g7e.2xlarge`, NVIDIA RTX PRO 6000 Blackwell 96 GB) —
  **NOT deployable yet on this stack** (see below). The selection cell stops
  with an explanation if you pick it.

### Why g7e can't run here yet

g7e is a Blackwell GPU (`sm_120`) and needs **CUDA 12.8+** kernels. This
notebook deploys a `PyTorchModel`, which resolves a **SageMaker PyTorch
*inference* Deep Learning Container (DLC)**. As of now the newest PyTorch
inference DLC AWS publishes is **2.6.0 (CUDA 12.4)** — it has no Blackwell
kernels, so the model would load but every generation fails at runtime with
`CUDA error: no kernel image is available for execution on the device`.

There is currently **no PyTorch *inference* DLC ≥ 2.7 / CUDA ≥ 12.8**. (PyTorch
*training* DLCs reach 2.9 / CUDA 13.0, but training images are not the
inference serving stack, and the pinned `sagemaker<3.0.0` SDK's `image_uris`
registry only knows inference framework versions up to 2.6.0 anyway.) AWS's
recommended G7e serving path today is the native vLLM / LMI container, which
targets LLMs and does not host a diffusers image pipeline like FLUX.2. Building
a custom Blackwell container is out of scope for this notebook.

### What to do once AWS ships Blackwell inference support

When a PyTorch **inference** DLC with **CUDA ≥ 12.8** is published (watch the
[available images list](https://aws.github.io/deep-learning-containers/reference/available_images/)):

1. Upgrade the SDK in section 1 so its `image_uris` registry knows the new
   version (unpin or raise `sagemaker<3.0.0`).
2. In the selection cell below, set g7e's `framework_version` / `py_version`
   to the new DLC (e.g. `"2.8"` / `"py312"`) and flip `_G7E_SUPPORTED = True`.
3. Re-run sections 3–4 to deploy. `inference.py` already branches on
   `BOOTH_INSTANCE_FAMILY=g7e` to keep the pipeline resident on CUDA
   (no CPU offload), so no handler change is needed.

In [ ]:
# Ask which GPU instance family to deploy on (g6e or g7e).
# Falls back to a sensible default in non-interactive runs.
#
# Each family pins the SageMaker PyTorch *inference* DLC version, because the
# GPU architecture must match the torch build:
#   - g6e = NVIDIA L40S (Ada, sm_89). The PyTorch 2.5 inference DLC supports it.
#   - g7e = NVIDIA RTX PRO 6000 Blackwell (sm_120, CUDA 12.9). Needs CUDA 12.8+
#     kernels. The NEWEST PyTorch *inference* DLC AWS publishes today is 2.6.0
#     (CUDA 12.4) -> no Blackwell kernels -> generations fail at runtime with
#     "CUDA error: no kernel image is available for execution on the device".
#     So g7e is NOT deployable on this PyTorchModel stack yet; we stop with an
#     explanation instead of shipping an endpoint that crash-loops on every
#     request. See the markdown above for the upgrade steps once AWS ships a
#     PyTorch inference DLC with CUDA >= 12.8.
_VALID = {
    "g6e": {"instance_type": "ml.g6e.2xlarge", "framework_version": "2.5", "py_version": "py311"},
    # g7e is kept here for the day AWS ships a Blackwell-capable inference DLC.
    # framework_version/py_version are placeholders; update them to the new DLC
    # (and flip _G7E_SUPPORTED below) when that lands.
    "g7e": {"instance_type": "ml.g7e.2xlarge", "framework_version": "2.7", "py_version": "py312"},
}

# Flip to True ONLY once a PyTorch *inference* DLC with CUDA >= 12.8 exists AND
# the SDK in section 1 has been upgraded to know that framework_version.
_G7E_SUPPORTED = False

try:
    _ans = input("Instance family to deploy on? [g6e/g7e] (default g6e): ").strip().lower()
except (EOFError, OSError):
    _ans = ""

instance_family = _ans if _ans in _VALID else "g6e"

if instance_family == "g7e" and not _G7E_SUPPORTED:
    raise SystemExit(
        "\n"
        "================================================================\n"
        " g7e (NVIDIA RTX PRO 6000 Blackwell) is NOT deployable yet.\n"
        "================================================================\n"
        "Why:\n"
        "  Blackwell GPUs (sm_120) need CUDA 12.8+ kernels. This notebook\n"
        "  deploys a PyTorchModel, which uses a SageMaker PyTorch *inference*\n"
        "  Deep Learning Container. The newest one AWS publishes today is\n"
        "  PyTorch 2.6.0 / CUDA 12.4 -- it has no Blackwell kernels, so the\n"
        "  model would load but every generation fails with:\n"
        "    'CUDA error: no kernel image is available for execution on the device'\n"
        "\n"
        "What to do once AWS ships support:\n"
        "  1. Confirm a PyTorch *inference* DLC with CUDA >= 12.8 exists:\n"
        "     https://aws.github.io/deep-learning-containers/reference/available_images/\n"
        "  2. Upgrade the SDK in section 1 (unpin / raise 'sagemaker<3.0.0') so\n"
        "     image_uris knows the new framework_version.\n"
        "  3. Set g7e's framework_version/py_version above to the new DLC and\n"
        "     flip _G7E_SUPPORTED = True, then re-run this cell.\n"
        "  (inference.py already handles g7e: it keeps the pipeline resident on\n"
        "   CUDA with no CPU offload when BOOTH_INSTANCE_FAMILY=g7e.)\n"
        "\n"
        "For now, deploy on g6e (re-run this cell and choose g6e).\n"
    )

_sel = _VALID[instance_family]
instance_type = _sel["instance_type"]
framework_version = _sel["framework_version"]
py_version = _sel["py_version"]

print(f"instance_family:   {instance_family}")
print(f"instance_type:     {instance_type}")
print(f"framework_version: {framework_version}  (PyTorch DLC)")
print(f"py_version:        {py_version}")

## 2. Configuration


In [ ]:
import os
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

# Stable prefix where the prepared weights will live. This is referenced by
# every deploy of this endpoint, so don't put a timestamp in here.
weights_prefix = "flux2-klein/weights"
weights_s3_uri = f"s3://{bucket}/{weights_prefix}/"


# Endpoint name carries the instance family suffix (chosen in section 1b) so
# g6e and g7e deployments are distinct, selectable endpoints in the booth.
endpoint_name = f"FluxAiBoot-{instance_family}"

print(f"region:           {region}")
print(f"bucket:           {bucket}")
print(f"weights_s3_uri:   {weights_s3_uri}")
print(f"instance_family:  {instance_family}")
print(f"instance_type:    {instance_type}")
print(f"endpoint_name:    {endpoint_name}")


## 3. Stage the weights + code to S3 (run once for weights)

`prepare_weights.py` downloads the bf16 FLUX.2 [klein] 9B repo from the Hub into a local directory in standard diffusers format, then `aws s3 sync` pushes it to the stable prefix above. We also upload the `code/` folder into the **same** prefix so the inference toolkit can find the handler at `/opt/ml/model/code/inference.py`.

The weight download is the **only** step that needs `HF_TOKEN`. Once the prefix is populated, redeploying never touches the Hub again.

Skip the weight download on subsequent deploys. When you change `code/inference.py`, you only need to re-run the `code/` upload cell — it's a few KB, no weight re-archiving. To verify the prefix is ready:
```bash
aws s3 ls --recursive s3://<bucket>/flux2-klein/weights/ --human-readable --summarize
```
Expected: ~33 GB across `model_index.json`, `scheduler/`, `text_encoder/`, `tokenizer/`, `transformer/`, `vae/`, and a small `code/` folder.


In [ ]:
# Run the prep script. Requires HF_TOKEN and access to the gated repo.
# Comment this out on subsequent deploys.

# hf_transfer is the only extra dep needed by the prep script. It speeds up
# the ~33 GB Hub download by ~5x. huggingface_hub is pre-installed on
# SageMaker notebook images.
# !pip install -q hf_transfer
# !HF_HUB_ENABLE_HF_TRANSFER=1 python prepare_weights.py --out weights

In [ ]:
# Sync the prepared weights to S3. `aws s3 sync` is idempotent — re-running
# only uploads files that have changed, which makes recovery from a
# partial run cheap.
# Comment this out on subsequent deploys (weights don't change).

# !aws s3 sync weights/ {weights_s3_uri}

In [ ]:
# Upload the handler into a code/ subfolder INSIDE the model prefix.
# This is the cell to re-run after editing code/inference.py — it's a few KB,
# no weight re-archiving. The inference toolkit loads /opt/ml/model/code/inference.py.

!aws s3 cp code/inference.py {weights_s3_uri}code/inference.py
!aws s3 cp code/requirements.txt {weights_s3_uri}code/requirements.txt

# Verify both files actually landed under the code/ prefix the container reads.
print("\nContents of code/ prefix (container reads /opt/ml/model/code/):")
!aws s3 ls {weights_s3_uri}code/

## 4. Deploy the endpoint

**Key wiring:**
- `model_data` is a `dict` describing the S3 prefix with `CompressionType: None`. SageMaker downloads every object under that prefix to `/opt/ml/model/` on the container without unpacking anything.
- The handler (`inference.py` + `requirements.txt`) lives in a `code/` subfolder **inside that same prefix** (uploaded in section 3). We deliberately do NOT pass `entry_point` / `source_dir` to `PyTorchModel`: with a dict `model_data`, those trigger the SDK's repack path, which doesn't support dict model data and silently skips wiring the code. Instead the SageMaker inference toolkit auto-discovers `/opt/ml/model/code/inference.py` via `SAGEMAKER_PROGRAM` + `SAGEMAKER_SUBMIT_DIRECTORY`.
- Editing the handler later means re-uploading just the small `code/` objects with `aws s3 cp` — never re-archiving the weights.
- No `HF_TOKEN` in the env. The container never talks to the Hub.
- `container_startup_health_check_timeout=3600` is the max. First cold start still has to `pip install` deps and load 34 GB, but pinning the diffusers release wheel (no from-source build) keeps it well under that.

**Latency expectation with CPU offload:** ~7–12 seconds per request on g6e.2xlarge. The 4-step diffusion takes about 1 second on the L40S, but offload moves ~34 GB of weights between CPU and GPU during each request. For batch / async workloads this is fine. For interactive UI workloads you'd want bnb 8-bit pre-quantized weights instead (different prep script, requires CUDA-extension dependency in the DLC, deferred to a follow-up).


In [ ]:
from sagemaker.pytorch import PyTorchModel
from sagemaker.async_inference import AsyncInferenceConfig

async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/flux2-klein-outputs/",
    failure_path=f"s3://{bucket}/flux2-klein-failures/",
    max_concurrent_invocations_per_instance=1,
)

# Uncompressed-prefix model data: SageMaker mirrors every object under the
# prefix to /opt/ml/model/ verbatim, no tar/gz round-trip on either side.
model_data = {
    "S3DataSource": {
        "S3Uri":           weights_s3_uri,
        "S3DataType":      "S3Prefix",
        "CompressionType": "None",
    }
}

# NOTE: with uncompressed-prefix model_data, do NOT pass entry_point /
# source_dir. Those trigger the SDK's repack path, which explicitly does
# not support a dict model_data (it logs a warning and silently skips
# wiring up the code). Instead, inference.py + requirements.txt live in a
# code/ subfolder INSIDE the model prefix (uploaded in section 3). The
# SageMaker inference toolkit auto-discovers /opt/ml/model/code/inference.py.
# The booth recreates an endpoint from a config whose name EQUALS the
# endpoint name (CreateEndpoint(EndpointName=cfg, EndpointConfigName=cfg)).
# The SageMaker SDK would otherwise auto-generate a timestamped config
# name; pin it to endpoint_name so the config is selectable in the booth.
endpoint_config_name = endpoint_name
print(f"endpoint_config_name: {endpoint_config_name}")

model = PyTorchModel(
    role=role,
    model_data=model_data,        # prefix containing weights + code/ subfolder
    framework_version=framework_version,  # pinned per instance family (g6e=2.5, g7e>=2.7)
    py_version=py_version,
    env={
        # Generous worker timeout so model_fn can finish loading ~34 GB.
        "SAGEMAKER_MODEL_SERVER_TIMEOUT": "3600",
        "TS_DEFAULT_RESPONSE_TIMEOUT":    "3600",
        # Belt-and-suspenders: the container hardcodes /opt/ml/model/code,
        # but set these too so intent is explicit.
        "SAGEMAKER_PROGRAM":           "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY":  "/opt/ml/model/code",
        # Tells inference.py which memory strategy to use:
        #   g6e -> enable_model_cpu_offload(); g7e -> pipe.to('cuda').
        "BOOTH_INSTANCE_FAMILY":       instance_family,
    },
)

# Clean slate: if an endpoint of this name already exists (e.g. a previous
# crash-looping attempt), delete it so we don't redeploy on top of a stale
# model/config. Redeploying to an existing name does NOT reliably replace
# the underlying model artifacts.
import boto3
sm = boto3.client("sagemaker")
for _name in [endpoint_name]:
    try:
        sm.delete_endpoint(EndpointName=_name)
        print(f"Deleted existing endpoint {_name}")
    except sm.exceptions.ClientError:
        pass
    try:
        sm.delete_endpoint_config(EndpointConfigName=_name)
        print(f"Deleted existing endpoint config {_name}")
    except sm.exceptions.ClientError:
        pass

predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type=instance_type,
    async_inference_config=async_config,
    endpoint_config_name=endpoint_config_name,
    # First cold start = pip install pinned deps + load 34 GB + offload
    # setup. The diffusers release wheel (not a git build) keeps this fast,
    # but use the max timeout for headroom.
    container_startup_health_check_timeout=3600,
    model_data_download_timeout=3600,
)

### If the deploy hangs or fails

The cell above blocks until the endpoint reaches `InService` or `Failed`. If it sits longer than expected, run the cell below from a fresh notebook cell (or a separate terminal) while the deploy is still in flight to tail the container's CloudWatch logs.


In [ ]:
# Tail container logs while the deploy is in progress.
# !aws logs tail /aws/sagemaker/Endpoints/{endpoint_name} --follow --since 1h

# Inspect the failure reason if the endpoint went to Failed:
# !aws sagemaker describe-endpoint --endpoint-name {endpoint_name} --query FailureReason --output text

## 5. Helpers for invoking the endpoint


In [ ]:
import base64
import io
import json
import time

import boto3
from PIL import Image

s3 = boto3.client("s3")
sm_runtime = boto3.client("sagemaker-runtime")


def encode_image(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def invoke(payload: dict, label: str) -> bytes:
    """Upload payload to S3, invoke the async endpoint, poll for the PNG."""
    input_key = f"flux2-klein-inputs/{label}-{int(time.time() * 1000)}.json"
    s3.put_object(
        Bucket=bucket,
        Key=input_key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )
    input_uri = f"s3://{bucket}/{input_key}"
    print(f"[{label}] input uploaded to {input_uri}")

    response = sm_runtime.invoke_endpoint_async(
        EndpointName=predictor.endpoint_name,
        InputLocation=input_uri,
        ContentType="application/json",
    )
    output_uri = response["OutputLocation"]
    print(f"[{label}] output expected at {output_uri}")

    path = output_uri.replace("s3://", "")
    out_bucket, out_key = path.split("/", 1)

    started = time.time()
    while True:
        try:
            obj = s3.get_object(Bucket=out_bucket, Key=out_key)
            elapsed = time.time() - started
            print(f"[{label}] done in {elapsed:.1f}s")
            return obj["Body"].read()
        except s3.exceptions.NoSuchKey:
            time.sleep(2)


## 6. Text-to-image

Distilled defaults: `num_inference_steps=4`, `guidance_scale=1.0`. Default output is 1024×1024; pass `height` / `width` (snapped to multiples of 16 internally) to override.


In [ ]:
from IPython.display import display

t2i_payload = {
    "inputs": (
        "A cinematic close-up of a snow leopard perched on a moss-covered rock at dawn, "
        "backlit by golden light filtering through pine needles, fur dusted with frost, "
        "shallow depth of field, photorealistic, 35mm film grain."
    ),
    "num_inference_steps": 4,
    "guidance_scale": 1.0,
    "height": 1024,
    "width": 1024,
    "seed": 42,
}

t2i_bytes = invoke(t2i_payload, label="t2i")
t2i_image = Image.open(io.BytesIO(t2i_bytes))
display(t2i_image.resize((512, 512), Image.LANCZOS))
print(f"Original: {t2i_image.width}x{t2i_image.height}")

## 7. Single-reference image editing

Pass `images` as a list of one base64-encoded PNG/JPEG. Output dimensions default to the reference image's.


In [ ]:
reference_b64 = encode_image(t2i_image)

edit_payload = {
    "inputs": (
        "Same composition and subject, but transformed into a stylized oil painting "
        "with thick impasto brush strokes, warm golden palette, and visible canvas texture."
    ),
    "images": [reference_b64],
    "num_inference_steps": 4,
    "guidance_scale": 1.0,
    "seed": 7,
}

edit_bytes = invoke(edit_payload, label="edit")
edit_image = Image.open(io.BytesIO(edit_bytes))
display(edit_image.resize((512, 512), Image.LANCZOS))
print(f"Original: {edit_image.width}x{edit_image.height}")

### 7b. Single-reference editing from a local image file

Set `LOCAL_IMAGE_PATH` to an image in the notebook's local folder (or any path on the notebook instance). Supports PNG, JPEG, and WebP. The cell loads it, shows the original, edits it with `EDIT_PROMPT`, and displays the result.


In [ ]:
# ---- edit these two ----
LOCAL_IMAGE_PATH = "my_image.webp"   # PNG, JPEG, or WebP
EDIT_PROMPT = "Turn this into a watercolor painting with soft pastel colors."
# ------------------------

import os
from PIL import features as _pil_features

if not os.path.exists(LOCAL_IMAGE_PATH):
    raise FileNotFoundError(
        f"{LOCAL_IMAGE_PATH!r} not found. Put an image next to this notebook "
        f"or set LOCAL_IMAGE_PATH to a full path. Files here: {os.listdir('.')[:20]}"
    )

# WebP needs Pillow built with WebP support. Most modern Pillow wheels have it,
# but check up front so the error is actionable rather than a cryptic decode fail.
if LOCAL_IMAGE_PATH.lower().endswith(".webp") and not _pil_features.check("webp"):
    raise RuntimeError(
        "This Pillow build has no WebP support. Install a newer Pillow: "
        "`pip install -U pillow` (the manylinux wheels bundle libwebp)."
    )

_loaded = Image.open(LOCAL_IMAGE_PATH)
# Animated WebP: take the first frame.
if getattr(_loaded, "is_animated", False):
    print(f"{LOCAL_IMAGE_PATH} is animated ({_loaded.n_frames} frames); using frame 0.")
    _loaded.seek(0)
# Flatten alpha (RGBA/LA/P-with-transparency) onto white, then force RGB.
if _loaded.mode in ("RGBA", "LA") or (_loaded.mode == "P" and "transparency" in _loaded.info):
    _rgba = _loaded.convert("RGBA")
    _bg = Image.new("RGB", _rgba.size, (255, 255, 255))
    _bg.paste(_rgba, mask=_rgba.split()[-1])
    source_image = _bg
else:
    source_image = _loaded.convert("RGB")

print(f"Loaded {LOCAL_IMAGE_PATH}: {source_image.width}x{source_image.height}")
print("Source image:")
display(source_image.resize((384, int(384 * source_image.height / source_image.width)), Image.LANCZOS))

# encode_image() re-encodes to PNG for transport (lossless), so the server
# receives a clean PNG regardless of the original WebP/JPEG source format.
local_edit_payload = {
    "inputs": EDIT_PROMPT,
    "images": [encode_image(source_image)],
    "num_inference_steps": 4,
    "guidance_scale": 1.0,
    "seed": 0,
}

local_edit_bytes = invoke(local_edit_payload, label="edit-local")
local_edit_image = Image.open(io.BytesIO(local_edit_bytes))
print("Edited result:")
display(local_edit_image.resize((384, int(384 * local_edit_image.height / local_edit_image.width)), Image.LANCZOS))
print(f"Result: {local_edit_image.width}x{local_edit_image.height}")

# Save the result next to the notebook.
_out = f"edited_{os.path.splitext(os.path.basename(LOCAL_IMAGE_PATH))[0]}.png"
local_edit_image.save(_out)
print(f"Saved to {_out}")

## 8. Multi-reference editing

Up to 4 reference images. The model composes the subject and style cues from the references under prompt guidance.


In [ ]:
multi_payload = {
    "inputs": (
        "Combine the subject from the first reference with the painterly style of the second, "
        "placed inside an autumn forest at golden hour."
    ),
    "images": [
        encode_image(t2i_image),
        encode_image(edit_image),
    ],
    "num_inference_steps": 4,
    "guidance_scale": 1.0,
    "seed": 123,
}

multi_bytes = invoke(multi_payload, label="multi")
multi_image = Image.open(io.BytesIO(multi_bytes))
display(multi_image.resize((512, 512), Image.LANCZOS))
print(f"Original: {multi_image.width}x{multi_image.height}")

## 9. Redeploying after a code change

When you edit `code/inference.py`:
1. Re-run the `code/` upload cell in section 3 (`aws s3 cp` of the two small files). Weights are untouched.
2. Re-run section 4 to recreate the endpoint against the same prefix.

Because the handler is plain objects in the model prefix (not a `sourcedir.tar.gz`), there's no archiving step at all — you overwrite `code/inference.py` in S3 directly. Cold start is bounded by EBS throughput, typically 2–3 minutes for ~33 GB.

Note: the container only reads the handler at startup, so an in-place `aws s3 cp` does not affect a running endpoint. You must redeploy (or `update_endpoint`) for code changes to take effect.


## 10. Cleanup


In [ ]:
predictor.delete_endpoint()
sagemaker.Session().delete_model(model.name)